# Deepfake Voice Detection — Live Detection
### Step 4 of 4: Real-time microphone detection using the trained model
---

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'sounddevice'], capture_output=True)
print('sounddevice installed successfully!')

## Step 1 — Import Libraries

In [ ]:
import os
import numpy as np
import librosa
import sounddevice as sd
import pickle

print('Libraries imported successfully!')

## Step 2 — Load Paths

In [ ]:
# AUTO PATH DETECTION — works on any machine
BASE_DIR           = os.path.dirname(os.path.abspath('__file__'))
REAL_CHUNKS_FOLDER = os.path.join(BASE_DIR, 'dataset', 'real_chunks')
FAKE_CHUNKS_FOLDER = os.path.join(BASE_DIR, 'dataset', 'fake_chunks')
MODELS_DIR         = os.path.join(BASE_DIR, 'models')

MODEL_PATH  = os.path.join(MODELS_DIR, 'model.pkl')
SCALER_PATH = os.path.join(MODELS_DIR, 'scaler.pkl')

print(f'Project directory : {BASE_DIR}')
print(f'Models directory  : {MODELS_DIR}')

## Step 3 — Load Model and Scaler from models/ Folder

In [ ]:
with open(MODEL_PATH, 'rb') as f:
    model = pickle.load(f)

with open(SCALER_PATH, 'rb') as f:
    scaler = pickle.load(f)

print(f'Model loaded  : {MODEL_PATH}')
print(f'Scaler loaded : {SCALER_PATH}')
print('Model and scaler ready!')

## Step 4 — Load Chunk Files from Dataset

In [ ]:
real_chunk_files = sorted([f for f in os.listdir(REAL_CHUNKS_FOLDER) if f.endswith('.wav')])
fake_chunk_files = sorted([f for f in os.listdir(FAKE_CHUNKS_FOLDER) if f.endswith('.wav')])

print(f'Real chunk files : {len(real_chunk_files)}')
print(f'Fake chunk files : {len(fake_chunk_files)}')
print(f'Total chunks     : {len(real_chunk_files) + len(fake_chunk_files)}')
print('Chunk files loaded. Ready for live detection.')

## Step 5 — live_detection() Function
Records audio from the microphone in 3-second chunks and predicts each one in real time.
Press Ctrl+C (or the stop button in Jupyter) to stop detection.

In [ ]:
def live_detection(sample_rate=16000, chunk_duration=3):
    '''
    Record audio from the microphone in 3-second chunks and predict
    whether each chunk is Real (human voice) or Fake (AI generated).
    Press Ctrl+C to stop.
    '''
    samples_per_chunk = int(sample_rate * chunk_duration)
    chunk_num         = 0

    print('=' * 55)
    print('     LIVE DEEPFAKE VOICE DETECTION')
    print('=' * 55)
    print(f'Sample rate    : {sample_rate} Hz')
    print(f'Chunk duration : {chunk_duration} seconds')
    print('Speak into your microphone. Press Ctrl+C to stop.')
    print()

    try:
        while True:
            chunk_num += 1
            print(f'[Chunk {chunk_num}] Recording {chunk_duration} seconds...', end=' ', flush=True)

            audio = sd.rec(
                samples_per_chunk,
                samplerate=sample_rate,
                channels=1,
                dtype='float32'
            )
            sd.wait()
            audio = audio.flatten()

            mfcc            = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
            features        = np.mean(mfcc, axis=1).reshape(1, -1)
            features_scaled = scaler.transform(features)

            prediction    = model.predict(features_scaled)[0]
            probabilities = model.predict_proba(features_scaled)[0]
            conf_pct      = probabilities[prediction] * 100
            label         = 'FAKE' if prediction == 1 else 'REAL'

            if label == 'FAKE':
                print(f'FAKE  (AI Generated Voice) — {conf_pct:.1f}% confident')
            else:
                print(f'REAL  (Human Voice)        — {conf_pct:.1f}% confident')

    except KeyboardInterrupt:
        print(f'\nLive detection stopped after {chunk_num} chunks.')
        print('Session complete.')

print('live_detection() function defined.')
print('Run the next cell to start live detection from your microphone.')

## Step 6 — Run Live Detection
Make sure your microphone is connected. Speak and the model will classify in real time.

In [ ]:
# Start live detection
# Speak into the microphone — each 3-second chunk is classified automatically
# Press Ctrl+C or the stop button in Jupyter to end the session

live_detection()